In [3]:
#matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler

train_set = pd.read_csv('data/train_set.csv')
features = ['laser energy', 'oxygen pressure', 'deposition_temp', 'TD_distance']
train_X = train_set[features]
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(train_X)
X_normalized = pd.DataFrame(X_normalized, columns=train_X.columns)
X_train = X_normalized
y_train = train_set['Jc_6']

In [4]:
class TreeNode:
    def __init__(self, feature_idx=None, left_bound=None, right_bound=None, 
                 left=None, right=None, value=None, a_opt=None, 
                 combo_feat_idx=None, raw_candidate_feat=None):
        self.feature_idx = feature_idx                # 分裂涉及的原始候选特征索引
        self.left_bound = left_bound                  # 区间左端点
        self.right_bound = right_bound                # 区间右端点
        self.left = left                              # 左子树（区间内） 
        self.right = right                            # 右子树（区间外）
        self.value = value                            # 叶节点预测值
        self.a_opt = a_opt                            # 线性组合权重（仅针对「最新组合特征+候选原始特征」）
        self.combo_feat_idx = combo_feat_idx          # 最新组合新特征的索引（拼接后在X中的位置）
        self.raw_candidate_feat = raw_candidate_feat  # 当前参与组合的原始候选特征

# 2. CART类
class CART:
    def __init__(self, task="classification", max_depth=5, min_samples_split=2, lambda_=0.1):
        self.task = task
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.lambda_ = lambda_       # 岭回归正则项
        self.root = None             # 树根节点
        self.init_n_features = None  # 原始特征总维度（固定）

    # 分类树：基尼系数
    def _gini(self, y):
        if len(y) == 0:
            return 0
        y = y.astype(int)
        class_counts = np.bincount(y)
        probabilities = class_counts / len(y)
        gini = 1 - np.sum(probabilities ** 2)
        return gini

    # 回归树：MSE
    def _mse(self, y):
        if len(y) == 0:
            return 0
        return np.mean((y - np.mean(y)) ** 2)

    # 统一计算分裂前损失
    def _base_loss(self, y):
        if self.task == "classification":
            return self._gini(y)
        else:
            return self._mse(y)

    # 求解「最新组合特征+候选原始特征」的最优线性组合权重
    def _get_linear_weights(self, X, y, candidate_feat, combo_feat_idx):
        # 构建特征子集：最新组合特征 + 当前候选原始特征（仅两个特征，避免冗余）
        used_feature_indices = [combo_feat_idx, candidate_feat]
        X_used = X[:, used_feature_indices]
        
        # 岭回归求解最优线性组合（无截距，避免偏移）
        ridge = Ridge(alpha=self.lambda_, fit_intercept=False)
        try:
            ridge.fit(X_used, y)
            return ridge.coef_
        except:
            return None

    # 重构：最优分裂查找（修复状态共享问题，接收子树独立状态参数）
    def _best_split(self, X, y, depth, remaining_raw_features, latest_combo_feat_idx):
        n_samples, n_features = X.shape
        best_feature_idx = None 
        best_left_bound = None
        best_right_bound = None
        best_a_opt = None               # 最优线性组合权重（仅两个特征：组合新特征+候选原始特征）
        best_gain = -np.inf             # 梯度提升（损失降低量）
        base_loss = self._base_loss(y)  # 分裂前基础损失

        # ---------------------- 第一层：单一原始特征区间分割（depth=0）----------------------
        if depth == 0:
            for feature_idx in range(self.init_n_features):  # 仅遍历原始特征，不涉及拼接特征
                X_feature = X[:, feature_idx].copy()
                unique_vals = np.unique(X_feature)
                sorted_vals = np.sort(unique_vals)

                for i in range(len(sorted_vals)):
                    left_bound = sorted_vals[i]
                    for j in range(i+1, len(sorted_vals)):
                        right_bound = sorted_vals[j]
                        # 区间内/外样本掩码
                        in_mask = (X_feature >= left_bound) & (X_feature <= right_bound)
                        out_mask = ~in_mask
                        y_in, y_out = y[in_mask], y[out_mask]

                        # 跳过空子集
                        if len(y_in) == 0 or len(y_out) == 0:
                            continue

                        # 计算加权损失
                        if self.task == "classification":
                            loss = (len(y_in)*self._gini(y_in) + len(y_out)*self._gini(y_out)) / len(y)
                        else:
                            loss = (len(y_in)*self._mse(y_in) + len(y_out)*self._mse(y_out)) / len(y)

                        # 计算梯度提升（损失降低越多，gain越大）
                        current_gain = base_loss - loss

                        # 更新最优分裂（优先选择gain更大的）
                        if current_gain > best_gain:
                            best_gain = current_gain
                            best_feature_idx = feature_idx
                            best_left_bound = left_bound
                            best_right_bound = right_bound

            return best_feature_idx, best_left_bound, best_right_bound, best_a_opt

        # ---------------------- 第二层及以后：组合新特征+候选原始特征 区间分割（depth≥1）----------------------
        else:
            if not remaining_raw_features:           # 无剩余原始特征，无法分裂
                return None, None, None, None
            if latest_combo_feat_idx is None:        # 还未生成组合新特征（仅depth=1时触发）
                return None, None, None, None
            if latest_combo_feat_idx >= n_features:  # 组合特征索引超出当前X的列数
                return None, None, None, None

            # 遍历剩余原始特征作为候选特征
            for candidate_feat in remaining_raw_features:
                # 1. 求解最优线性组合权重（最新组合特征 + 候选原始特征）
                a_opt = self._get_linear_weights(X, y, candidate_feat, latest_combo_feat_idx)
                if a_opt is None:
                    continue

                # 2. 计算线性组合的投影得分（即新的组合特征值，保存为后续使用）
                used_feature_indices = [latest_combo_feat_idx, candidate_feat]
                X_used = X[:, used_feature_indices]
                proj_scores = X_used @ a_opt  # 这就是要保存的新组合特征

                # 3. 对新组合特征（投影得分）做区间分割
                unique_scores = np.unique(proj_scores)
                sorted_scores = np.sort(unique_scores)

                for i in range(len(sorted_scores)):
                    left_bound = sorted_scores[i]
                    for j in range(i+1, len(sorted_scores)):
                        right_bound = sorted_scores[j]
                        # 区间内/外样本掩码
                        in_mask = (proj_scores >= left_bound) & (proj_scores <= right_bound)
                        out_mask = ~in_mask
                        y_in, y_out = y[in_mask], y[out_mask]

                        # 跳过空子集
                        if len(y_in) == 0 or len(y_out) == 0:
                            continue

                        # 4. 计算加权损失和梯度提升
                        if self.task == "classification":
                            loss = (len(y_in)*self._gini(y_in) + len(y_out)*self._gini(y_out)) / len(y)
                        else:
                            loss = (len(y_in)*self._mse(y_in) + len(y_out)*self._mse(y_out)) / len(y)

                        current_gain = base_loss - loss

                        # 5. 更新最优分裂（梯度提升最大）
                        if current_gain > best_gain:
                            best_gain = current_gain
                            best_feature_idx = candidate_feat
                            best_left_bound = left_bound
                            best_right_bound = right_bound
                            best_a_opt = a_opt

            return best_feature_idx, best_left_bound, best_right_bound, best_a_opt

    # 叶节点预测值
    def _leaf_value(self, y):
        if self.task == "classification":
            return np.argmax(np.bincount(y.astype(int)))  # 众数
        else:
            return np.mean(y)  # 均值

    # 拼接组合新特征到特征矩阵（保存为最新组合特征，返回新矩阵和新索引）
    def _append_combo_feat(self, X, proj_scores):
        # 将投影得分作为新特征拼接至X末尾，返回新矩阵和最新组合特征索引
        X_new = np.hstack([X, proj_scores.reshape(-1, 1)])
        new_combo_feat_idx = X_new.shape[1] - 1
        return X_new, new_combo_feat_idx

    # 构建树（传递子树独立状态，避免全局共享）
    def _build_tree(self, X, y, depth, remaining_raw_features, latest_combo_feat_idx):
        n_samples = len(y)

        # 停止条件：深度/样本数/纯度足够
        stop_conditions = [
            depth >= self.max_depth,
            n_samples < self.min_samples_split,
            self._base_loss(y) < 1e-6  # 纯度足够，无需分裂
        ]
        if any(stop_conditions):
            return TreeNode(value=self._leaf_value(y))

        # 查找最优分裂（传递子树独立状态）
        feat_idx, left_bound, right_bound, a_opt = self._best_split(
            X, y, depth, remaining_raw_features, latest_combo_feat_idx
        )
        if feat_idx is None:  # 无有效分裂
            return TreeNode(value=self._leaf_value(y))

        # ---------------------- 第一层：处理单一原始特征分裂，生成第一个组合新特征 ----------------------
        if depth == 0:
            # 1. 按单一特征区间分割数据集
            X_feature = X[:, feat_idx].copy()
            in_mask = (X_feature >= left_bound) & (X_feature <= right_bound)
            out_mask = ~in_mask
            X_in, y_in = X[in_mask], y[in_mask]
            X_out, y_out = X[out_mask], y[out_mask]

            # 2. 生成第一个组合新特征（仅当前最优单一特征，为后续层做准备）
            proj_scores_in = X_in[:, feat_idx]
            proj_scores_out = X_out[:, feat_idx]

            # 3. 拼接组合新特征到子树的特征矩阵中（获取子树独立的组合特征索引）
            X_in_new, combo_idx_in = self._append_combo_feat(X_in, proj_scores_in)
            X_out_new, combo_idx_out = self._append_combo_feat(X_out, proj_scores_out)

            # 4. 生成子树独立的剩余原始特征（移除当前已选原始特征，不修改全局）
            remaining_raw_in = [f for f in remaining_raw_features if f != feat_idx]
            remaining_raw_out = [f for f in remaining_raw_features if f != feat_idx]

            # 5. 记录当前节点信息（第一层无线性组合权重）
            current_node = TreeNode(
                feature_idx=feat_idx,
                left_bound=left_bound,
                right_bound=right_bound,
                combo_feat_idx=latest_combo_feat_idx  # 第一层无前置组合特征，传None
            )

        # ---------------------- 第二层及以后：处理组合新特征+候选原始特征分裂 ----------------------
        else:
            if a_opt is None or latest_combo_feat_idx is None:
                return TreeNode(value=self._leaf_value(y))

            # 1. 计算当前最优线性组合的投影得分（新的组合特征）
            used_feature_indices = [latest_combo_feat_idx, feat_idx]
            X_used = X[:, used_feature_indices]
            proj_scores = X_used @ a_opt

            # 2. 按投影得分区间分割数据集
            in_mask = (proj_scores >= left_bound) & (proj_scores <= right_bound)
            out_mask = ~in_mask
            X_in, y_in = X[in_mask], y[in_mask]
            X_out, y_out = X[out_mask], y[out_mask]
            proj_scores_in = proj_scores[in_mask]
            proj_scores_out = proj_scores[out_mask]

            # 3. 拼接新的组合特征到子树的特征矩阵中（获取子树独立的组合特征索引）
            X_in_new, combo_idx_in = self._append_combo_feat(X_in, proj_scores_in)
            X_out_new, combo_idx_out = self._append_combo_feat(X_out, proj_scores_out)

            # 4. 生成子树独立的剩余原始特征（移除当前已选原始特征，不修改全局）
            remaining_raw_in = [f for f in remaining_raw_features if f != feat_idx]
            remaining_raw_out = [f for f in remaining_raw_features if f != feat_idx]

            # 5. 记录当前节点信息（包含线性组合权重和组合新特征索引）
            current_node = TreeNode(
                feature_idx=feat_idx,
                left_bound=left_bound,
                right_bound=right_bound,
                a_opt=a_opt,
                combo_feat_idx=latest_combo_feat_idx,
                raw_candidate_feat=feat_idx
            )

        # ---------------------- 递归构建左右子树（传递子树独立的状态参数）----------------------
        left_node = self._build_tree(
            X_in_new, y_in, depth + 1, remaining_raw_in, combo_idx_in
        )
        right_node = self._build_tree(
            X_out_new, y_out, depth + 1, remaining_raw_out, combo_idx_out
        )

        current_node.left = left_node
        current_node.right = right_node

        return current_node

    # 重构：训练方法（初始化子树独立状态，不再使用全局状态）
    def fit(self, X, y):
        X = np.array(X, dtype=np.float64)
        y = np.array(y)
        self.init_n_features = X.shape[1]
        # 初始化子树独立状态（剩余原始特征、初始组合特征索引）
        remaining_raw_features = list(range(self.init_n_features))
        latest_combo_feat_idx = None  # 初始无组合特征
        # 构建树（传递初始独立状态）
        self.root = self._build_tree(
            X, y, depth=0,
            remaining_raw_features=remaining_raw_features,
            latest_combo_feat_idx=latest_combo_feat_idx
        )

    # 重构：单样本预测（适配子树独立组合特征，逐层还原拼接）
    def _predict_sample(self, x, node):
        """单样本预测：逐层还原组合新特征，判断区间归属"""
        if node.value is not None:  # 叶节点，返回预测值
            return node.value

        # 提取节点信息
        feat_idx = node.feature_idx
        left_bound = node.left_bound
        right_bound = node.right_bound
        a_opt = node.a_opt
        combo_feat_idx = node.combo_feat_idx

        # ---------------------- 第一层节点：单一原始特征判断 + 生成初始组合特征 ----------------------
        if a_opt is None:
            feat_val = x[feat_idx]
            # 区间判断
            if left_bound <= feat_val <= right_bound:
                # 拼接初始组合特征（单一特征值），继续预测左子树
                x_new = np.hstack([x, np.array([feat_val])])
                return self._predict_sample(x_new, node.left)
            else:
                # 拼接初始组合特征（单一特征值），继续预测右子树
                x_new = np.hstack([x, np.array([feat_val])])
                return self._predict_sample(x_new, node.right)

        # ---------------------- 后续层节点：组合新特征+候选原始特征 判断 ----------------------
        else:
            if combo_feat_idx is None or combo_feat_idx >= len(x):  # 索引越界兜底
                return self._predict_sample(x, node.right)

            # 1. 提取用于组合的两个特征：最新组合特征 + 候选原始特征
            used_feature_vals = np.array([x[combo_feat_idx], x[feat_idx]])
            # 2. 计算新的组合特征得分（投影得分）
            proj_score = used_feature_vals @ a_opt
            # 3. 拼接新的组合特征到样本向量中（更新为最新）
            x_new = np.hstack([x, np.array([proj_score])])
            # 4. 区间判断
            if left_bound <= proj_score <= right_bound:
                return self._predict_sample(x_new, node.left)
            else:
                return self._predict_sample(x_new, node.right)

    # 批量预测
    def predict(self, X):
        X = np.array(X, dtype=np.float64)
        return np.array([self._predict_sample(x, self.root) for x in X])